# Sensor placement optimization + Isaac Lab (Colab)

This notebook is the **practical** Colab entrypoint for this repository when you use the unofficial Isaac Sim / Isaac Lab Colab installer workflow (same idea as `isaac_lab_2_1_0_colab.ipynb` in this folder).

**What you get end-to-end**
- Installs Isaac Sim + Isaac Lab (same script pattern as the uploaded Colab)
- Clones *this* repo to `/content/sensor_placement_opt`
- Launches a small **HTTP JSON bridge**: `scripts/isaaclab_sensor_bridge.py` (`/health`, `/reconfigure_sensors`, `/run_rollouts`)
- Runs CMA-ES in `sensor_opt` via `IsaacSimEvaluator` over `http://127.0.0.1:<port>`

**Bridge modes (`--bridge-mode`)**
- **`ground`** — `blind_spot_fraction` from depth / point-like observations (with fallbacks) + best-effort collision / success from `info`.
- **`obstacle`** — static obstacle corridor: spawns 3–5 boxes per reset, reports **`t_det_s` / `t_det_s_p95`**, contact-based **collision rate**, and **safety** fields. Use with `configs/obstacle_isaaclab.yaml` (`loss.mode: obstacle_latency`).

**What you should still set for your robot / task**
- In `configs/*.yaml`, set `sensor_models.<lidar|camera>.isaac.prim_path` so `_apply_sensor_config` can move sensor prims (use `{env_idx}` for cloned envs if needed).
- Set `ISAAC_TASK` to an environment ID your Isaac Lab build actually registers.


## Configure your 3D LiDAR + RGB-D mounts in the USD stage (recommended)

`SensorConfig` already describes each sensor (type, mounting slot, offsets, FoV fractions). To actually *move* sensors in Isaac, set USD prim paths in your YAML (e.g. `configs/default.yaml` or `configs/obstacle_isaaclab.yaml`) under:
- `sensor_models.lidar.isaac.prim_path`
- `sensor_models.camera.isaac.prim_path`

For parallel env cloning, paths often look like:
`/World/envs/env_{env_idx}/Robot/.../YourLidar`
`/World/envs/env_{env_idx}/Robot/.../YourDepthCamera`

The bridge will `.format(env_idx=...)` if `{env_idx}` is present.

## Metrics and configs

- **`--bridge-mode ground`** — `blind_spot_fraction` from **depth** + **3D point-like** arrays in observations (task-dependent). Falls back to `fast_baseline_metrics(...)` for that episode if perception is missing (one-time warning).
- **`--bridge-mode obstacle`** — research-style corridor metrics: `t_det_s_p95`, `collision_rate`, `safety_success` / `mean_goal_success` (no-contact + clearance). Pair with **`configs/obstacle_isaaclab.yaml`** and `inner_loop.n_episodes` / `max_steps_per_episode` consistent with the bridge’s `--max-steps` and `--sim-dt`.


In [4]:
# Install Isaac Sim 4.5 and Isaac Lab 2.1.0. This process takes about 10 mins to complete.
!wget -O install-isaac-sim-4.5.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-sim-4.5.sh
!time bash install-isaac-sim-4.5.sh
!wget -O install-isaac-lab-2.1.0.sh https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-lab-2.1.0.sh
!time bash install-isaac-lab-2.1.0.sh


--2026-04-24 14:23:30--  https://raw.githubusercontent.com/j3soon/isaac-sim-colab/main/scripts/install-isaac-sim-4.5.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4517 (4.4K) [text/plain]
Saving to: ‘install-isaac-sim-4.5.sh’

install-isaac-sim-4 100%[===================>]   4.41K  --.-KB/s    in 0s      

2026-04-24 14:23:30 (51.1 MB/s) - ‘install-isaac-sim-4.5.sh’ saved [4517/4517]

--2026-04-24 14:23:30--  https://raw.githubusercontent.com/j3soon/colab-python-version/main/scripts/py310.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 2

In [ ]:
# Clone *this* repository (the sensor placement optimizer) and install its Python deps.
# NOTE: if you are iterating on a fork, change REPO_URL.

import os
import subprocess
import sys

REPO_URL = "https://github.com/dcaglar-28/sensor_placement_opt.git"
REPO_DIR = "/content/sensor_placement_opt"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # IMPORTANT: in Colab, use `{REPO_DIR}` (Python formatting), not `"$REPO_DIR"` in `!` shells.
    !rm -rf "{REPO_DIR}"
    !git clone "{REPO_URL}" "{REPO_DIR}"

os.chdir(REPO_DIR)

# Install into the *same interpreter as this kernel* (avoids "pip installed but import fails")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

print("OK:", os.getcwd())
print("python:", sys.executable)


## Interactive: set sensor USD prims (run before the bridge)

The bridge moves each sensor `Xform` using your YAML and/or these env vars, and re-applies after every `env.reset()`. Without valid **prim paths**, every CMA-ES candidate looks the same in sim.

Options: set `sensor_models.*.isaac.prim_path` in your YAML, or run the next cell to set **`ISAAC_LIDAR_PRIM`**, **`ISAAC_CAMERA_PRIM`**, **`ISAAC_RADAR_PRIM`** (support `{env_idx}` in the string for vectorized envs). Optional: **`CONFIG_PATH`** (defaults in the optimizer cell if unset).


In [1]:
import os

def _ask(name: str, current: str = "") -> str:
    try:
        v = input(f"{name} [{current}]: ").strip()
    except EOFError:
        v = ""
    return v or current

print("(Press Enter to skip and use YAML / defaults.)")
L = _ask("ISAAC_LIDAR_PRIM", os.environ.get("ISAAC_LIDAR_PRIM", ""))
C = _ask("ISAAC_CAMERA_PRIM", os.environ.get("ISAAC_CAMERA_PRIM", ""))
R = _ask("ISAAC_RADAR_PRIM", os.environ.get("ISAAC_RADAR_PRIM", ""))
P = _ask("CONFIG_PATH", os.environ.get("CONFIG_PATH", ""))
if L: os.environ["ISAAC_LIDAR_PRIM"] = L
if C: os.environ["ISAAC_CAMERA_PRIM"] = C
if R: os.environ["ISAAC_RADAR_PRIM"] = R
if P: os.environ["CONFIG_PATH"] = P
print("ISAAC_LIDAR_PRIM=", os.environ.get("ISAAC_LIDAR_PRIM", "(unset)"))
print("ISAAC_CAMERA_PRIM=", os.environ.get("ISAAC_CAMERA_PRIM", "(unset)"))
print("ISAAC_RADAR_PRIM=", os.environ.get("ISAAC_RADAR_PRIM", "(unset)"))
print("CONFIG_PATH=", os.environ.get("CONFIG_PATH", "(use optimizer default)"))


(Press Enter to skip and use YAML / defaults.)
ISAAC_LIDAR_PRIM []: /World/envs/env_{env_idx}/Vehicle/Sensors/LidarTop
ISAAC_CAMERA_PRIM []: /World/envs/env_{env_idx}/Vehicle/Sensors/FrontCamera
ISAAC_RADAR_PRIM []: /World/envs/env_{env_idx}/Vehicle/Sensors/FrontRadar
CONFIG_PATH []: configs/obstacle_isaaclab.yaml
ISAAC_LIDAR_PRIM= /World/envs/env_{env_idx}/Vehicle/Sensors/LidarTop
ISAAC_CAMERA_PRIM= /World/envs/env_{env_idx}/Vehicle/Sensors/FrontCamera
ISAAC_RADAR_PRIM= /World/envs/env_{env_idx}/Vehicle/Sensors/FrontRadar
CONFIG_PATH= configs/obstacle_isaaclab.yaml


## Start the Isaac Lab HTTP bridge (background process)

`scripts/isaaclab_sensor_bridge.py` follows the Isaac Lab pattern: `AppLauncher` → `parse_env_cfg` → `gym.make` → `env.step`.

The next cell starts the bridge with **`subprocess` + this kernel’s `sys.executable`** (reliable in Colab). Set **`BRIDGE_MODE`** to `ground` or `obstacle` (or `os.environ["BRIDGE_MODE"]`). For **`obstacle`**, optional `D_WARN`, `D_CLEAR`, `SIM_DT`, and `MAX_STEPS` are passed through.

`PORT` and `NUM_ENVS` must match the optimizer cell below.

If you get import errors, re-run the Isaac install cell and use the same Python the installer expects.


In [1]:
REPO_DIR = "/content/sensor_placement_opt"  # add this line first
ISAAC_PYTHON = None                          # then discover python below

import glob, os, subprocess, sys, time, urllib.error, urllib.request

candidates = [
    "/isaac-sim/python.sh",
    "/isaac-sim/kit/python/bin/python3",
    "/root/.local/share/ov/pkg/isaac-sim-*/python.sh",
    "/usr/local/bin/python",
]
for pattern in candidates:
    for p in glob.glob(pattern):
        if os.path.exists(p):
            ISAAC_PYTHON = p
            break
    if ISAAC_PYTHON:
        break

print("REPO_DIR:", REPO_DIR)
print("ISAAC_PYTHON:", ISAAC_PYTHON)
os.chdir(REPO_DIR)
print("FOUND:", p)

REPO_DIR: /content/sensor_placement_opt
ISAAC_PYTHON: /usr/local/bin/python
FOUND: /usr/local/bin/python


In [2]:
os.chdir(REPO_DIR)

# Isaac Sim / Kit environment flags (per Isaac Sim install docs)
os.environ["OMNI_KIT_ACCEPT_EULA"] = "YES"

PORT = 8010
TASK = os.environ.get("ISAAC_TASK", "Isaac-Velocity-Flat-Unitree-Go2-v0")
NUM_ENVS = 1
BRIDGE_MODE = os.environ.get("BRIDGE_MODE", "ground")  # "ground" | "obstacle" | "generic"
MAX_STEPS = 500

# Video recording (optional)
RECORD_VIDEO = os.environ.get("RECORD_VIDEO", "1") not in ("0", "false", "False")
VIDEO_DIR = os.environ.get("ISAAC_VIDEO_DIR", "/tmp/isaaclab_video")
if RECORD_VIDEO:
    os.makedirs(VIDEO_DIR, exist_ok=True)
    os.environ["ISAAC_VIDEO_DIR"] = VIDEO_DIR

# Obstacle-mode extras (ignored when BRIDGE_MODE != "obstacle")
D_WARN = 3.0
D_CLEAR = 0.5
SIM_DT = 0.02
OBSTACLE_ROOT = "/World/bridge_corridor"

BRIDGE = os.path.join(REPO_DIR, "scripts", "isaaclab_sensor_bridge.py")
LOG = "/tmp/isaaclab_sensor_bridge.log"
ISAAC_PYTHON = "/usr/local/bin/python"

!pkill -f "isaaclab_sensor_bridge.py" 2>/dev/null || true

cmd = [
    ISAAC_PYTHON,
    BRIDGE,
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--bridge-mode", BRIDGE_MODE,
    "--headless",
    "--task", TASK,
    "--num_envs", str(NUM_ENVS),
    "--repo-root", REPO_DIR,
    "--max-steps", str(MAX_STEPS),
]

if RECORD_VIDEO:
    cmd += [
        "--video",
        "--video_length", str(int(os.environ.get("VIDEO_LENGTH", "200"))),
        "--video_interval", str(int(os.environ.get("VIDEO_INTERVAL", "1"))),
    ]

if BRIDGE_MODE == "obstacle":
    cmd += [
        "--d-warn", str(D_WARN),
        "--d-clear", str(D_CLEAR),
        "--sim-dt", str(SIM_DT),
        "--obstacle-root", OBSTACLE_ROOT,
    ]

_log = open(LOG, "wb")
_proc = subprocess.Popen(
    cmd,
    stdout=_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
_log.close()
print("Launched bridge pid=", _proc.pid, "mode=", BRIDGE_MODE, "log=", LOG, flush=True)
if RECORD_VIDEO:
    print("Video dir:", VIDEO_DIR, flush=True)

# The bridge only binds HTTP after `AppLauncher` + `gym.make` init; that can take many minutes in Colab.
deadline = time.time() + 60 * 30
last_log_pos = 0


def _read_new_log_bytes() -> bytes:
    global last_log_pos
    try:
        with open(LOG, "rb") as f:
            f.seek(last_log_pos)
            chunk = f.read()
            last_log_pos = f.tell()
            return chunk
    except FileNotFoundError:
        return b""


print("Waiting for HTTP bridge: http://127.0.0.1:%d/health ..." % PORT, flush=True)
while time.time() < deadline:
    chunk = _read_new_log_bytes()
    if chunk:
        print(chunk.decode("utf-8", errors="replace"), end="")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
            body = r.read()
        print("Bridge is ready:", body.decode("utf-8", errors="replace").strip(), flush=True)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError, OSError):
        time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for bridge /health. See log: " + LOG)

!tail -n 60 "{LOG}" || true
print("Bridge log:", LOG)


^C
Launched bridge pid= 19347 mode= ground log= /tmp/isaaclab_sensor_bridge.log
Waiting for HTTP bridge: http://127.0.0.1:8010/health ...
[Warning] [simulation_app.simulation_app] Modules: ['omni.kit_app'] were loaded before SimulationApp was started and might not be loaded correctly.
[Warning] [simulation_app.simulation_app] Please check to make sure no extra omniverse or pxr modules are imported before the call to SimulationApp(...)
[Warning] [simulation_app.simulation_app] fast shutdown not supported with jupyter notebooks
Loading user config located at: '/usr/local/lib/python3.10/site-packages/omni/data/Kit/Isaac-Sim/4.5/user.config.json'
[Info] [carb] Logging to file: /usr/local/lib/python3.10/site-packages/omni/logs/Kit/Isaac-Sim/4.5/kit_20260424_150158.log
2026-04-24 15:01:58 s] [Warning] [omni.kit.app.plugin] No crash reporter present, dumps uploading isn't available.
2026-04-24 15:01:59 s] [Warning] [omni.ext.plugin] [ext: rendering_modes] Extensions config 'extension.toml' do

## Run the sensor placement optimizer against the local bridge

The JSON response from `/run_rollouts` includes all `EvalMetrics` fields (see `sensor_opt/loss/loss.py`). The client (`sensor_opt.inner_loop.bridge_json_client.BridgeJsonClient`) maps every key so **`t_det_s_p95`**, **`safety_success`**, etc. are not dropped when using `--bridge-mode obstacle`.

By default this cell loads **`configs/default.yaml`** and runs a **tiny** CMA smoke (1 generation, small population). Set **`CONFIG_PATH`** to **`configs/obstacle_isaaclab.yaml`** when `BRIDGE_MODE=obstacle` so `loss.mode: obstacle_latency` matches the bridge metrics.

**Shared sensor noise (all trial YAMLs):** set the same value under `inner_loop.isaac_sim.sensor_noise_std` (meters, i.i.d. on range-style observations in the bridge). The optimizer passes it on every `/run_rollouts` call so you do not need to restart the bridge when changing the YAML.

**Convergence figure:** CMA-ES logs each generation to `results/<run_id>/generations.csv`. The cell below draws an **SVG** (NumPy only, no matplotlib) and shows it with `IPython.display.SVG`: **best in generation**, **best-so-far**, and **mean ± std** (classic optimization paper view).


In [3]:
import copy
import glob
import os

import numpy as np
import yaml
from IPython.display import SVG, Video, display

from sensor_opt.config.catalog import apply_sensor_catalog
from sensor_opt.inner_loop.bridge_json_client import BridgeJsonClient
from sensor_opt.inner_loop.isaac_evaluator import IsaacSimEvaluator
from sensor_opt.logging.experiment_logger import ExperimentLogger
from sensor_opt.plotting.convergence import plot_convergence_from_csv
from sensor_opt.search.factory import create_search


def show_latest_video(video_dir: str | None = None) -> None:
    video_dir = video_dir or os.environ.get("ISAAC_VIDEO_DIR", "/tmp/isaaclab_video")
    patterns = [
        os.path.join(video_dir, "**", "*.mp4"),
        "/content/IsaacLab/logs/**/*.mp4",
        "/content/sensor_placement_opt/logs/**/*.mp4",
    ]
    vids = []
    for p in patterns:
        vids.extend(glob.glob(p, recursive=True))
    if not vids:
        print("No videos found yet.")
        return
    latest = max(vids, key=os.path.getmtime)
    print("Showing latest video:", latest)
    display(Video(latest, embed=True))


os.chdir("/content/sensor_placement_opt")

PORT = 8010  # must match the bridge cell
NUM_ENVS = 1  # must match --num_envs in the bridge cell
CONFIG_PATH = os.environ.get("CONFIG_PATH", "configs/default.yaml")

_cfg0 = yaml.safe_load(open(CONFIG_PATH))
_cfg0 = apply_sensor_catalog(_cfg0)
sensor_noise_std = float(_cfg0.get("inner_loop", {}).get("isaac_sim", {}).get("sensor_noise_std", 0.0) or 0.0)

bridge_env = BridgeJsonClient(base_url=f"http://127.0.0.1:{PORT}", timeout_sec=60 * 30)
base_evaluator = IsaacSimEvaluator(
    isaac_sim_cfg={
        "env": bridge_env,
        "num_envs": NUM_ENVS,
        "sensor_noise_std": sensor_noise_std,
    }
)

cfg = copy.deepcopy(_cfg0)
cfg["inner_loop"]["mode"] = "isaac_sim"
cfg["multi_fidelity"]["enabled"] = False
print("sensor_noise_std (m):", sensor_noise_std)

# tiny smoke settings
cfg["cma"]["max_generations"] = 1
cfg["cma"]["population_size"] = 4
cfg["cma"]["tolx"] = 1e-9
cfg["cma"]["tolfun"] = 1e-9

seed = int(cfg.get("experiment", {}).get("seed", 42))

with ExperimentLogger(
    experiment_name=cfg["experiment"]["name"] + "_isaaclab_smoke",
    results_dir=cfg.get("logging", {}).get("results_dir", "results"),
    use_mlflow=False,
    mlflow_tracking_uri=cfg.get("logging", {}).get("mlflow_tracking_uri", "mlruns"),
    run_config=cfg,
) as logger:
    search = create_search(
        search_type=cfg["search"]["type"],
        config=cfg,
        evaluator={
            "evaluator_fn": None,
            "evaluator": None,
            "base_evaluator": base_evaluator,
            "logger": logger,
            "seed": seed,
        },
    )
    result = search.run()
    generations_csv = str(logger.run_dir / "generations.csv")

print("config:", CONFIG_PATH)
print("best_loss:", result.best_loss)
print("run_id:", result.run_id)
print("generations.csv:", generations_csv)

# Academic-style convergence (SVG — no matplotlib; works in Colab / Jupyter)
try:
    _svg = plot_convergence_from_csv(generations_csv, title=f"Convergence — {result.run_id}")
    display(SVG(_svg))
except Exception as _e:
    print("Convergence plot skipped (check generations.csv):", _e)

show_latest_video()


[CMA-ES] Vector dimension: 50 (5 sensor slots × 10 params)


KeyboardInterrupt: 

### Core paper figures (SVG, no matplotlib)

After a full CMA-ES run, the run directory contains:

| File | Use |
|------|-----|
| `generations.csv` | §1 multi-curve convergence, §3 σ vs generation, §11 sample efficiency |
| `evaluated_pool.json` | §7 correlation, §6 parameter histograms (all candidates) |
| `pareto_front.json` | §2 Pareto scatter (lead figure for multi-objective) |
| `optimization_meta.json` | population size, total evals |
| `final_result.json` | §4 top-down layout (`best_config`) |

**Priority (tight page limit):** §2 Pareto → §1 convergence → §4 layout → §9 baselines (hand-built table). Use `sensor_opt.plotting.paper_figures` for §1–§3, §5–§7, §9–§11. §8 CDF needs per-episode `t_det` you log yourself. §4 uses schematic wedges (FoV from YAML `horizontal_fov_deg`).

The next cell shows example `display(SVG(...))` calls; point paths at `run_dir` (parent of `generations.csv`).


In [ ]:
# Paper figures (run after the optimizer cell so `generations_csv` / `result` / `cfg` exist)
from pathlib import Path

import json
from IPython.display import SVG, display

from sensor_opt.plotting.paper_figures import (
    fig01_convergence_multi,
    fig02_pareto_scatter_2d,
    fig03_cma_sigma,
    fig04_topdown_sensors,
    fig05_slot_heatmap,
    fig06_param_distributions,
    fig07_correlation_heatmap,
    fig08_cdf,
    fig09_baseline_bars,
    fig10_hypervolume_vs_budget,
    fig11_sample_efficiency,
)

run_dir = Path(generations_csv).parent

# §1 Multi-curve convergence (overlay several runs: list of (generations.csv path, label))
display(SVG(fig01_convergence_multi([(generations_csv, "this_run")])))

# σ vs generation (exploration vs exploitation)
display(SVG(fig03_cma_sigma(generations_csv)))

# Pareto scatter: adjust x_key/y_key for obstacle_latency (e.g. collision vs blind_spot proxy) vs default mode
try:
    pf_path = run_dir / "pareto_front.json"
    if pf_path.is_file():
        display(SVG(fig02_pareto_scatter_2d(pf_path, x_key="collision", y_key="blind_spot")))
except Exception as e:
    print("Pareto figure:", e)

# Correlation across all evaluated candidates
try:
    pool_path = run_dir / "evaluated_pool.json"
    if pool_path.is_file():
        display(SVG(fig07_correlation_heatmap(pool_path)))
except Exception as e:
    print("Correlation figure:", e)

# Best layout (schematic); pass sensor_models from config for FoV
try:
    fr_path = run_dir / "final_result.json"
    if fr_path.is_file():
        fr = json.loads(fr_path.read_text())
        display(SVG(fig04_topdown_sensors(fr.get("best_config") or {}, cfg.get("sensor_models"))))
except Exception as e:
    print("Layout figure:", e)

# Sample efficiency: best-so-far vs cumulative evaluations
display(SVG(fig11_sample_efficiency(generations_csv)))

# §5 Slot heatmap: `matrix[slot_name][goal_name] = "lidar"` (build from several runs)
# display(SVG(fig05_slot_heatmap({"s0": {"obstacle": "lidar", "ground": "radar"}}, title="Slot utilization")))

# §6 Parameter histogram (yaw, pitch, offsets): one `param` at a time, from evaluated_pool
# try:
#     if pool_path.is_file():
#         display(SVG(fig06_param_distributions(pool_path, param="yaw_deg")))
# except Exception as e:
#     print("Param distributions:", e)

# §8 CDF: pass per-episode t_det (s) samples (any order; sorted inside)
# display(SVG(fig08_cdf([0.1, 0.2, 0.3], label="t_det (s)")))

# §10 Hypervolume vs cost cap: list of (cost_cap_usd, hypervolume) from ablations
# display(SVG(fig10_hypervolume_vs_budget([(100.0, 0.05), (200.0, 0.12), (500.0, 0.18)])))

# Baseline comparison — fill with your hand-tuned / random baselines (metrics must match your reporting)
# display(SVG(fig09_baseline_bars({
#     "hand": {"collision": 0.2, "blind_spot": 0.3},
#     "optimized": {"collision": 0.05, "blind_spot": 0.1},
#     "random": {"collision": 0.35, "blind_spot": 0.4},
# })))
